# Project 1 — Supermarket Sales Analysis

**Objective:** Analyze sales, profitability, customer segments, products, regions, and transaction patterns using pandas and visualization.

**Dataset:** Superstore-style retail dataset (9,994 rows, 21 original columns).

### Business questions
1. How did sales and profit change over time?
2. Which categories and sub-categories contribute most to sales and profit?
3. Which regions and customer segments generate the most revenue?
4. Which days show the highest sales?
5. How is discount associated with profit?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/supermarket_sales_clean.csv")
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])

print("Shape:", df.shape)
df.head()


## 1. Data Quality Validation

In [ ]:
quality = pd.DataFrame({
    "Metric": ["Rows", "Columns", "Missing cells", "Duplicate rows", "Negative sales", "Negative profit rows"],
    "Value": [
        len(df), len(df.columns), int(df.isna().sum().sum()),
        int(df.duplicated().sum()), int((df["Sales"] < 0).sum()),
        int((df["Profit"] < 0).sum())
    ]
})
quality


**Validation result:** The original dataset contains no missing values and no duplicate rows. Negative profit is retained because it represents real loss-making transactions and is analytically important.

## 2. KPI Summary

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
orders = df["Order ID"].nunique()
customers = df["Customer ID"].nunique()

kpis = {
    "Total Sales": total_sales,
    "Total Profit": total_profit,
    "Total Quantity": df["Quantity"].sum(),
    "Unique Orders": orders,
    "Unique Customers": customers,
    "Average Order Value": total_sales / orders,
    "Profit Margin %": total_profit / total_sales * 100
}
pd.Series(kpis)


## 3. Annual Trend

In [ ]:
annual = df.groupby(df["Order Date"].dt.year).agg(
    Sales=("Sales","sum"),
    Profit=("Profit","sum"),
    Orders=("Order ID","nunique")
)
annual


In [ ]:
annual["Sales"].plot(marker="o", figsize=(9,5), title="Annual Sales Trend")
plt.xlabel("Year")
plt.ylabel("Sales")
plt.show()


## 4. Category and Sub-Category Analysis

In [ ]:
category = df.groupby("Category").agg(
    Sales=("Sales","sum"),
    Profit=("Profit","sum"),
    Quantity=("Quantity","sum")
).sort_values("Sales", ascending=False)

category


In [ ]:
category[["Sales","Profit"]].plot(kind="bar", figsize=(9,5), title="Sales and Profit by Category")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.show()


In [ ]:
subcategory = df.groupby("Sub-Category").agg(
    Sales=("Sales","sum"),
    Profit=("Profit","sum"),
    Quantity=("Quantity","sum")
).sort_values("Sales", ascending=False)

subcategory.head(10)


## 5. Regional and Segment Analysis

In [ ]:
region = df.groupby("Region").agg(
    Sales=("Sales","sum"),
    Profit=("Profit","sum"),
    Orders=("Order ID","nunique")
).sort_values("Sales", ascending=False)

segment = df.groupby("Segment").agg(
    Sales=("Sales","sum"),
    Profit=("Profit","sum"),
    Orders=("Order ID","nunique")
).sort_values("Sales", ascending=False)

region, segment


In [ ]:
region["Sales"].plot(kind="bar", figsize=(9,5), title="Sales by Region")
plt.ylabel("Sales")
plt.xticks(rotation=0)
plt.show()


## 6. Day-of-Week Analysis

In [ ]:
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekday = df.assign(Weekday=df["Order Date"].dt.day_name()).groupby("Weekday").agg(
    Sales=("Sales","sum"),
    Profit=("Profit","sum")
).reindex(weekday_order)

weekday


In [ ]:
weekday["Sales"].plot(kind="bar", figsize=(10,5), title="Sales by Day of Week")
plt.ylabel("Sales")
plt.xticks(rotation=25)
plt.show()


## 7. Discount and Profit

In [ ]:
corr = df[["Sales","Quantity","Discount","Profit"]].corr()
corr


In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["Discount"], df["Profit"], alpha=0.25)
plt.axhline(0, linewidth=1)
plt.title("Discount vs Profit")
plt.xlabel("Discount")
plt.ylabel("Profit")
plt.show()


## 8. Key Findings

- The dataset covers January 2014 through December 2017 and contains 9,994 transaction rows.
- Technology generates the largest category sales total, while Office Supplies has the largest quantity sold.
- The West region has the highest sales total in the dataset.
- Sales rise substantially from 2014 to 2017.
- Some high-revenue sub-categories, particularly Tables and Bookcases, produce negative aggregate profit.
- Discount has a negative correlation with profit in this dataset; correlation indicates association, not causation.
- Loss-making transactions are numerous enough to warrant product/discount-level investigation.


## 9. Recommendations

1. Review pricing and discount practices for loss-making sub-categories, especially Tables and Bookcases.
2. Investigate the drivers of strong West-region performance and determine whether successful practices can be replicated elsewhere.
3. Prioritize profitable high-sales products while monitoring margin rather than revenue alone.
4. Segment discount analysis by product and customer segment before changing promotion policy.
5. Repeat the analysis monthly to detect emerging changes earlier.


## 10. Limitations

- The dataset does not provide acquisition cost, inventory levels, marketing spend, or customer satisfaction.
- Correlation does not establish causation.
- The analysis is descriptive and does not forecast future sales.
- Results apply to this dataset and period; they should not automatically be generalized to other retailers.
